# 04 — Tester le DPO et décider de ne pas le déployer

Je cherche à améliorer les réponses par préférences après le SFT. Deux essais ont été réalisés : un pilote généraliste puis un essai ciblé sur les erreurs du SFT mixte. **Aucun n'améliore suffisamment le triage pour être retenu.**
[03 · SFT](03_sft.ipynb) · [Parcours](README.md) · [05 · Déploiement](05_deploiement.ipynb)


## 1. Ce que le DPO apprend

```text
même demande → réponse préférée / réponse rejetée
             → optimiser le LoRA par rapport à une référence SFT figée
             → générer sur validation → mesurer les erreurs → accepter ou rejeter
```

Le SFT apprend à reproduire une réponse. Le DPO augmente la préférence relative pour une réponse plutôt qu'une autre, par rapport au modèle de référence. Une loss DPO plus faible ne prouve pas une amélioration médicale.

| Essai | Point de départ | Paires | Résultat utile |
|---|---|---|---|
| Généraliste | SFT successif v3 | Préférences natives UltraMedical ; pilote 256 train / 64 validation | Mêmes cinq urgences manquées, macro-F1 légèrement en baisse |
| Ciblé | SFT mixte époque 3 | Erreurs réelles sur le train triage ; 355 train / 85 contrôle | Moins de sur-triage, mais davantage d'urgences manquées |

Les poids finaux du premier pilote n'étaient pas disponibles pour le test indépendant. Ses métriques de validation sont conservées ; je ne prétends pas disposer de son évaluation sur test.


## 2. Construire l'essai ciblé sans entraîner sur la validation

1. [inference.py](../evaluation/inference.py) génère sur les **1 888 entrées du train triage** avec le SFT mixte.
2. [preparer_dpo_triage.py](../training/preparer_dpo_triage.py) garde les erreurs complètes et cohérentes : réponse préférée = référence silver ; rejetée = réponse du modèle. Il exclut les sorties invalides, tronquées et les ESI déjà corrects.
3. Les 440 paires sont séparées par cas, FR/EN groupés : **355 train, 85 contrôle interne**. Ce contrôle vient du train SFT ; il ne mesure pas une généralisation indépendante.
4. [entrainer_dpo.py](../training/entrainer_dpo.py) réalise **32 mises à jour**, taux 0,000002, beta 0,1, batch effectif 16, référence figée. Il sauvegarde les checkpoints 16 et 32.
5. L'inférence et les métriques sont calculées pour le SFT de référence et les deux checkpoints sur les mêmes validations QA et triage, dans la même session Colab.

Les neuf erreurs urgentes de validation observées après SFT ne sont pas réinjectées dans le train. Les références silver ne deviennent pas des labels cliniques par le DPO.


In [1]:
import json
from pathlib import Path

# Fonctionne depuis notebooks/ ou depuis la racine du dépôt.
RACINE = next(p for p in (Path.cwd(), *Path.cwd().parents)
              if (p / "serving/space/support/resultats.json").is_file())
def lire_json(relatif):
    return json.loads((RACINE / relatif).read_text(encoding="utf-8"))

resultats = lire_json("serving/space/support/resultats.json")
print("Source : bilans sauvegardés ; aucune inférence ni aucun entraînement relancé.")


Source : bilans sauvegardés ; aucune inférence ni aucun entraînement relancé.


## 3. Résultats : l'objectif de triage se dégrade

Je relis les rapports agrégés de la session ciblée. La référence mixte est réévaluée dans cette session ; ses petits écarts avec le carnet précédent ne proviennent pas d'un nouvel entraînement.


In [2]:
print(f"{'Modèle':<29} {'QCM':>8} {'Macro-F1':>10} {'Urgents':>10} {'Sur-triage':>12}")
for ligne in resultats["lignes"]:
    if ligne["groupe"] == "dpo_cible":
        s = ligne["scores"]
        print(f"{ligne['nom']:<29} {s['accuracy_qcm']:>7.2%} {s['macro_f1_priorite']:>10.4f} {s['rappel_maximale']:>9.0%} {s['taux_sur_triage']:>11.2%}")


Modèle                             QCM   Macro-F1    Urgents   Sur-triage
Référence mixte réévaluée      56.49%     0.7875       91%      18.49%
DPO ciblé · 16 pas             60.31%     0.7457       87%      14.71%
DPO ciblé · 32 pas             58.78%     0.7373       83%      12.18%


## 4. Décision et interprétation

Le rappel urgent passe de **91 % à 87 %, puis 83 %**. Les deux checkpoints échouent aux critères de rappel et de macro-F1 définis avant l'essai : **je conserve le SFT mixte**. Une amélioration des QCM ou une baisse du sur-triage ne compense pas ces urgences supplémentaires manquées.

Le train DPO contient 234 sur-triages, 72 sous-triages critiques et 49 autres sous-triages. Cette majorité de sur-triages peut orienter vers des niveaux moins urgents ; c'est une hypothèse explicative, pas une causalité démontrée. Les différences de longueur et de style des réponses constituent une autre limite.

Avant un nouvel essai, je ferais revoir les préférences et leur équilibre, puis prévoirais une confirmation indépendante. Aucun nouveau calcul n'est nécessaire pour constater le résultat négatif de cet essai.


## 5. Où sont les productions ?

Sous `data/colab_archive/dpo_triage_cible_20261106_132004/` :

| Fichier ou dossier | Contenu |
|---|---|
| `triage_train_minage.jsonl` | Réponses du SFT sur le train |
| `paires/` | Préférences retenues et partitionnement |
| `entrainement/` | Poids, protocole, résumé du DPO |
| `validation/` | Prédictions QA/triage et métriques des trois variantes |
| `comparaison_validation.json` | Bilan et décision |

Le [contrôle de récupération](../reports/bilan_dpo_cible_20261106.json) rapporte six rapports recalculés identiques sur 1 500 prédictions. Les journaux historiques (archives locales non publiées) conservent les lancements. Le carnet suivant présente le déploiement du SFT mixte conservé.
